# Fine-tune Llama 3.1 8B for Text-to-SQL

This notebook fine-tunes `meta-llama/Meta-Llama-3.1-8B-Instruct` on our text-to-SQL dataset using **LoRA** (Low-Rank Adaptation) via **Unsloth**.

**Runtime:** GPU required. In Google Colab, go to `Runtime → Change runtime type → T4 GPU` (free tier is enough).

**What LoRA does:**
Instead of retraining all 8B parameters (expensive, slow), LoRA freezes the original weights and trains small adapter matrices injected into each attention layer. This reduces trainable parameters by ~99% while achieving comparable performance on narrow tasks.

**Steps:**
1. Install Unsloth
2. Load base model with 4-bit quantization (fits on a T4)
3. Attach LoRA adapters
4. Upload and load our JSONL training data
5. Train with SFTTrainer
6. Save the adapter weights
7. Test the fine-tuned model
8. Export to Ollama-compatible GGUF format

## 1. Install Unsloth

Unsloth patches HuggingFace Transformers to be 2x faster and use 60% less VRAM. Essential for training on a free Colab T4.

In [ ]:
%%capture
# Install Unsloth — this takes ~2 minutes on Colab
!pip install unsloth
!pip install --upgrade --no-cache-dir unsloth

## 2. Load base model with 4-bit quantization

4-bit quantization (QLoRA) compresses the model weights from 32-bit floats to 4-bit integers, reducing VRAM from ~16GB to ~5GB. Quality loss is minimal for fine-tuning purposes.

> **Note:** You need a HuggingFace account and must accept Meta's Llama 3.1 license at https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048  # covers our longest schema + question + SQL
DTYPE = None           # auto-detect: float16 for T4/V100, bfloat16 for A100
LOAD_IN_4BIT = True    # QLoRA: 4-bit quantization

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct",  # Unsloth's pre-quantized version
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

print(f"Model loaded. Parameters: {model.num_parameters():,}")

## 3. Attach LoRA adapters

LoRA injects trainable rank-decomposition matrices into the attention layers (`q_proj`, `k_proj`, etc.). `r=16` is the rank — higher rank = more capacity but more parameters to train. `lora_alpha=16` is the scaling factor.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                     # LoRA rank — 16 is a good default for most tasks
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention projections
        "gate_proj", "up_proj", "down_proj",        # feed-forward layers
    ],
    lora_alpha=16,
    lora_dropout=0,           # 0 is optimal per Unsloth benchmarks
    bias="none",              # no bias training — saves parameters
    use_gradient_checkpointing="unsloth",  # reduces VRAM by ~30%
    random_state=42,
)

# Show trainable vs frozen parameter counts
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} ({100 * trainable / total:.2f}% of total)")

## 4. Upload and load training data

Upload the files from your local `datasets/finetune/` directory:
- `train.jsonl`
- `eval.jsonl`

Run `python scripts/prepare_finetune_data.py` locally first if you haven't already.

In [ ]:
from google.colab import files

print("Upload train.jsonl and eval.jsonl from datasets/finetune/ on your machine")
uploaded = files.upload()

In [ ]:
from datasets import load_dataset

train_dataset = load_dataset("json", data_files="train.jsonl", split="train")
eval_dataset  = load_dataset("json", data_files="eval.jsonl",  split="train")

print(f"Train examples: {len(train_dataset)}")
print(f"Eval examples:  {len(eval_dataset)}")
print("\nFirst training example messages:")
for msg in train_dataset[0]["messages"]:
    print(f"  [{msg['role'].upper()}] {msg['content'][:100]}...")

## 5. Train

We use HuggingFace TRL's `SFTTrainer` (Supervised Fine-Tuning Trainer) which handles the training loop.

**Key hyperparameters:**
- `num_train_epochs=3` — 3 full passes over the data. With 40 examples this is fast.
- `learning_rate=2e-4` — standard LoRA learning rate; higher than full fine-tuning because adapters start from zero.
- `per_device_train_batch_size=2` — small batch for T4 VRAM constraints.
- `gradient_accumulation_steps=4` — simulates a batch size of 8 without the memory cost.

**Important:** SFTTrainer expects a plain string field, not a list of message dicts. We apply the chat template first to convert `messages` → a single formatted string, then pass `dataset_text_field="text"`. Skipping this step causes a `TypeError: TextEncodeInput` error.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import get_chat_template

# Step 1: apply chat template to tokenizer so it knows Llama's special tokens
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

# Step 2: convert messages list → formatted string for each example
# SFTTrainer needs a plain string field, not a list of dicts
def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_dataset_fmt = train_dataset.map(format_example, num_proc=1)
eval_dataset_fmt  = eval_dataset.map(format_example,  num_proc=1)

print("Sample formatted text (first 300 chars):")
print(train_dataset_fmt[0]["text"][:300])

# Step 3: train
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset_fmt,
    eval_dataset=eval_dataset_fmt,
    dataset_text_field="text",        # plain string field
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=1,               # keep at 1 to avoid multiprocessing errors in Colab
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,     # effective batch size = 8
        num_train_epochs=3,
        warmup_steps=5,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="./text2sql-llama",
    ),
)

# Show VRAM usage before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"\nGPU: {gpu_stats.name}")
print(f"VRAM reserved: {start_gpu_memory} GB / {max_memory} GB total")

In [ ]:
# Train — expect ~10-20 minutes on a T4 for 3 epochs over 40 examples
trainer_stats = trainer.train()

used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"\nTraining complete.")
print(f"Peak VRAM: {used_memory} GB")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.1f}s")
print(f"Final train loss: {trainer_stats.metrics['train_loss']:.4f}")

## 6. Test the fine-tuned model

Run a few queries manually to get a feel for quality before saving.

In [ ]:
# tokenizer already has the llama-3.1 chat template from the training cell
FastLanguageModel.for_inference(model)  # enable 2x faster inference

SCHEMA = """customers(id, name, email, country, signup_date DATE)
products(id, name, category, price DECIMAL)
orders(id, customer_id, order_date DATE, status VARCHAR)  -- status: completed | pending | cancelled
order_items(id, order_id, product_id, quantity INTEGER, unit_price DECIMAL)"""

def ask(question: str) -> str:
    messages = [
        {"role": "system", "content": "You are an expert SQL engineer. Output ONLY the DuckDB SQL query, no explanation."},
        {"role": "user",   "content": f"Schema:\n{SCHEMA}\n\nQuestion: {question}"},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    outputs = model.generate(
        input_ids=inputs, max_new_tokens=256, temperature=1, use_cache=True
    )
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    return response.strip()

test_questions = [
    "How many customers are there in total?",
    "What is the total revenue from completed orders?",
    "Which customers have never placed a completed order?",
    "Who are the top 3 customers by total spend on completed orders?",
]

for q in test_questions:
    print(f"Q: {q}")
    print(f"A: {ask(q)}")
    print()

## 7. Save LoRA adapter weights

Save just the adapter (not the full 8B model). The adapter is ~50MB — much smaller than the 16GB base model.

In [ ]:
ADAPTER_PATH = "text2sql-llama-adapter"

model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"Adapter saved to {ADAPTER_PATH}/")

# Zip and download to your local machine
import shutil
shutil.make_archive(ADAPTER_PATH, "zip", ADAPTER_PATH)

from google.colab import files
files.download(f"{ADAPTER_PATH}.zip")

## 8. Export to GGUF for Ollama

GGUF is the format used by llama.cpp and Ollama for local inference. Exporting merges the LoRA adapter back into the base model and quantizes it to Q4_K_M (4-bit, ~4.5GB).

After downloading the GGUF file, create an Ollama model from it locally:

```bash
# Create a Modelfile
echo 'FROM ./text2sql-llama.Q4_K_M.gguf' > Modelfile

# Import into Ollama
ollama create text2sql-llama -f Modelfile

# Run it
ollama run text2sql-llama

# Evaluate it through the existing harness (LiteLLM supports Ollama natively)
python scripts/run_eval.py --model ollama/text2sql-llama --strategy zero_shot
```

In [ ]:
# Merge adapter into base model and export as GGUF Q4_K_M
# This takes ~5 minutes and requires ~10GB disk space
model.save_pretrained_gguf(
    "text2sql-llama",
    tokenizer,
    quantization_method="q4_k_m",  # good balance of size vs quality
)

print("GGUF export complete.")
!ls -lh text2sql-llama*.gguf

In [ ]:
# Download the GGUF file to your local machine
from google.colab import files
files.download("text2sql-llama.Q4_K_M.gguf")
print("\nNext steps:")
print("  echo 'FROM ./text2sql-llama.Q4_K_M.gguf' > Modelfile")
print("  ollama create text2sql-llama -f Modelfile")
print("  python scripts/run_eval.py --model ollama/text2sql-llama --strategy zero_shot")